# Notebook 06 — Hardware-Counter Overlays

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebooks 01–05 built a structural pipeline:

1. input distributions  
2. cache / branching proxies  
3. SIMD vs scalar execution paths  
4. constraint phase maps  
5. real benchmark ingestion  

Notebook 06 adds hardware-counter overlays:

- branch misses
- cache misses
- IPC
- cycle count
- stall / memory pressure proxies
- observed throughput and latency

Constraint view:
> performance becomes interpretable when structural predictions meet hardware counters.

## Goals

1. Look for hardware-counter files in:

```text
rml_extension/results/hardware_counters/
```

2. Normalize common counter columns:
   - branch_misses
   - branch_instructions
   - cache_misses
   - cache_references
   - cycles
   - instructions
   - elapsed_seconds
   - throughput_mib_s
   - latency_ns

3. Merge hardware counters with Notebook 05 benchmark-ingestion outputs.
4. Compute derived metrics:
   - branch_miss_rate
   - cache_miss_rate
   - IPC
   - cycles_per_item
   - counter_pressure_score
5. Produce overlay figures.

If no counter file is present, this notebook creates a transparent synthetic counter table so the workflow remains runnable.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
COUNTER_DIR = RESULTS_DIR / "hardware_counters"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, COUNTER_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)
print("COUNTER_DIR:", COUNTER_DIR)

## Load Notebook 05 benchmark table

Notebook 05 provides observed throughput/latency and phase metrics.

In [ ]:
bench_path = RESULTS_DIR / "notebook05_real_benchmark_ingestion.csv"

if bench_path.exists():
    bench = pd.read_csv(bench_path)
    print("Loaded:", bench_path)
else:
    print("Notebook 05 output not found; using fallback benchmark table.")
    bench = pd.DataFrame([
        {"distribution": "low_entropy_repeating", "observed_throughput_mib_s": 1650, "observed_latency_ns": 0.60, "coherence_score": 0.75, "fragmentation_score": 0.02, "regime": "coherent-local"},
        {"distribution": "sequential_ids", "observed_throughput_mib_s": 1350, "observed_latency_ns": 0.74, "coherence_score": 0.52, "fragmentation_score": 0.36, "regime": "scalar-favorable"},
        {"distribution": "uniform_32bit", "observed_throughput_mib_s": 1900, "observed_latency_ns": 0.53, "coherence_score": 0.24, "fragmentation_score": 0.95, "regime": "simd-favorable"},
        {"distribution": "zipfian_smallints", "observed_throughput_mib_s": 1500, "observed_latency_ns": 0.67, "coherence_score": 0.42, "fragmentation_score": 0.79, "regime": "simd-favorable"},
        {"distribution": "clustered_ranges", "observed_throughput_mib_s": 950, "observed_latency_ns": 1.05, "coherence_score": 0.22, "fragmentation_score": 1.00, "regime": "fragmented-irregular"},
    ])

bench.head()

## Ingest hardware-counter files

You can place CSV/JSON exports from `perf stat`, benchmark wrappers, or hand-normalized tables in:

```text
rml_extension/results/hardware_counters/
```

This notebook accepts flexible column names and normalizes them.

In [ ]:
def read_file(path):
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() == ".json":
        try:
            return pd.read_json(path)
        except ValueError:
            return pd.DataFrame(json.loads(path.read_text()))
    return None

frames = []
for path in sorted(COUNTER_DIR.glob("*")):
    if path.suffix.lower() not in [".csv", ".json"]:
        continue
    try:
        tmp = read_file(path)
        if tmp is not None and len(tmp):
            tmp["source_file"] = path.name
            frames.append(tmp)
            print("Loaded counter file:", path.name, tmp.shape)
    except Exception as e:
        print("Skipping", path.name, "because", e)

if frames:
    raw_counters = pd.concat(frames, ignore_index=True)
else:
    print("No hardware-counter files found; creating synthetic hardware-counter table.")
    raw_counters = pd.DataFrame([
        {"distribution": "low_entropy_repeating", "branch_misses": 1000, "branch_instructions": 2_000_000, "cache_misses": 20_000, "cache_references": 4_000_000, "cycles": 90_000_000, "instructions": 180_000_000, "items": 100_000},
        {"distribution": "sequential_ids", "branch_misses": 12_000, "branch_instructions": 2_500_000, "cache_misses": 55_000, "cache_references": 4_500_000, "cycles": 120_000_000, "instructions": 210_000_000, "items": 100_000},
        {"distribution": "uniform_32bit", "branch_misses": 80_000, "branch_instructions": 2_800_000, "cache_misses": 110_000, "cache_references": 5_000_000, "cycles": 100_000_000, "instructions": 260_000_000, "items": 100_000},
        {"distribution": "zipfian_smallints", "branch_misses": 95_000, "branch_instructions": 3_100_000, "cache_misses": 90_000, "cache_references": 4_800_000, "cycles": 130_000_000, "instructions": 240_000_000, "items": 100_000},
        {"distribution": "clustered_ranges", "branch_misses": 140_000, "branch_instructions": 3_000_000, "cache_misses": 190_000, "cache_references": 4_900_000, "cycles": 180_000_000, "instructions": 230_000_000, "items": 100_000},
    ])

raw_counters.head()

## Normalize counter columns

In [ ]:
def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def normalize_counters(raw):
    df = raw.copy()
    mappings = {
        "distribution": ["distribution", "input_distribution", "dataset", "name"],
        "branch_misses": ["branch_misses", "branches_missed", "branch-misses"],
        "branch_instructions": ["branch_instructions", "branches", "branch-instructions"],
        "cache_misses": ["cache_misses", "cache-misses", "llc_misses", "l1_misses"],
        "cache_references": ["cache_references", "cache-references", "cache_refs"],
        "cycles": ["cycles", "cpu_cycles", "cpu-cycles"],
        "instructions": ["instructions", "instr", "inst_retired"],
        "items": ["items", "n", "count", "sample_size"],
        "hardware_profile": ["hardware_profile", "hardware", "platform", "machine"],
        "implementation": ["implementation", "algorithm", "method", "function"],
    }

    out = pd.DataFrame()
    for target, cols in mappings.items():
        src = first_existing(df, cols)
        out[target] = df[src] if src else None

    out["distribution"] = out["distribution"].astype(str)
    for c in ["branch_misses", "branch_instructions", "cache_misses", "cache_references", "cycles", "instructions", "items"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out["hardware_profile"] = out["hardware_profile"].fillna("unknown").astype(str)
    out["implementation"] = out["implementation"].fillna("unknown").astype(str)

    return out.dropna(subset=["distribution"])

counters = normalize_counters(raw_counters)
counters

## Aggregate and derive hardware-counter metrics

In [ ]:
agg = (
    counters
    .groupby("distribution", as_index=False)
    .agg(
        branch_misses=("branch_misses", "mean"),
        branch_instructions=("branch_instructions", "mean"),
        cache_misses=("cache_misses", "mean"),
        cache_references=("cache_references", "mean"),
        cycles=("cycles", "mean"),
        instructions=("instructions", "mean"),
        items=("items", "mean"),
        hardware_profiles=("hardware_profile", lambda x: ", ".join(sorted(set(map(str, x))))),
        implementations=("implementation", lambda x: ", ".join(sorted(set(map(str, x))))),
    )
)

eps = 1e-12
agg["branch_miss_rate"] = agg["branch_misses"] / (agg["branch_instructions"] + eps)
agg["cache_miss_rate"] = agg["cache_misses"] / (agg["cache_references"] + eps)
agg["ipc"] = agg["instructions"] / (agg["cycles"] + eps)
agg["cycles_per_item"] = agg["cycles"] / (agg["items"] + eps)

def norm01(s):
    s = pd.Series(s).astype(float)
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)

agg["branch_miss_norm"] = norm01(agg["branch_miss_rate"])
agg["cache_miss_norm"] = norm01(agg["cache_miss_rate"])
agg["cycles_per_item_norm"] = norm01(agg["cycles_per_item"])
agg["ipc_norm"] = norm01(agg["ipc"])

agg["counter_pressure_score"] = (
    0.35 * agg["branch_miss_norm"] +
    0.35 * agg["cache_miss_norm"] +
    0.20 * agg["cycles_per_item_norm"] +
    0.10 * (1.0 - agg["ipc_norm"])
).clip(0, 1)

agg

## Merge counters with benchmark and phase metrics

In [ ]:
merged = bench.merge(agg, on="distribution", how="left")

# Optional disagreement metrics
if "fragmentation_score" in merged.columns:
    merged["fragmentation_counter_gap"] = merged["fragmentation_score"].fillna(0) - merged["counter_pressure_score"].fillna(0)
    merged["abs_fragmentation_counter_gap"] = merged["fragmentation_counter_gap"].abs()

if "observed_throughput_mib_s" in merged.columns:
    t = merged["observed_throughput_mib_s"].astype(float)
    if t.max() != t.min():
        merged["observed_throughput_norm"] = (t - t.min()) / (t.max() - t.min())
    else:
        merged["observed_throughput_norm"] = 0.0

merged

## Export overlay table

In [ ]:
csv_path = RESULTS_DIR / "notebook06_hardware_counter_overlays.csv"
json_path = RESULTS_DIR / "notebook06_hardware_counter_overlays.json"

merged.to_csv(csv_path, index=False)
merged.to_json(json_path, orient="records", indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)

## Figure 1 — Branch miss rate by distribution

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook06_branch_miss_rate.png"

plot_df = merged.sort_values("branch_miss_rate")
plt.figure(figsize=(9, 5))
plt.bar(plot_df["distribution"], plot_df["branch_miss_rate"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Branch miss rate")
plt.title("Hardware Counters: Branch Miss Rate")
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Cache miss rate by distribution

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook06_cache_miss_rate.png"

plot_df = merged.sort_values("cache_miss_rate")
plt.figure(figsize=(9, 5))
plt.bar(plot_df["distribution"], plot_df["cache_miss_rate"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Cache miss rate")
plt.title("Hardware Counters: Cache Miss Rate")
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Fragmentation proxy vs counter pressure

This is the key validation plot: do structural fragmentation proxies agree with hardware-counter pressure?

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook06_fragmentation_vs_counter_pressure.png"

plt.figure(figsize=(8, 6))
plt.scatter(merged["fragmentation_score"], merged["counter_pressure_score"])
for _, row in merged.iterrows():
    plt.annotate(row["distribution"], (row["fragmentation_score"], row["counter_pressure_score"]), fontsize=8)
plt.xlabel("Predicted fragmentation score")
plt.ylabel("Observed counter pressure score")
plt.title("Validation Overlay: Fragmentation vs Hardware-Counter Pressure")
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — IPC vs observed throughput

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook06_ipc_vs_throughput.png"

plt.figure(figsize=(8, 6))
plt.scatter(merged["ipc"], merged["observed_throughput_mib_s"])
for _, row in merged.iterrows():
    plt.annotate(row["distribution"], (row["ipc"], row["observed_throughput_mib_s"]), fontsize=8)
plt.xlabel("Instructions per cycle (IPC)")
plt.ylabel("Observed throughput (MiB/s)")
plt.title("Hardware Overlay: IPC vs Throughput")
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Counter summary matrix

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook06_counter_summary_matrix.png"

matrix_cols = [
    "branch_miss_norm",
    "cache_miss_norm",
    "cycles_per_item_norm",
    "ipc_norm",
    "counter_pressure_score",
    "observed_throughput_norm",
]

mat = merged.set_index("distribution")[matrix_cols].sort_values("counter_pressure_score", ascending=False)

plt.figure(figsize=(9, 5))
plt.imshow(mat.values, aspect="auto")
plt.yticks(range(len(mat.index)), mat.index)
plt.xticks(range(len(matrix_cols)), matrix_cols, rotation=45, ha="right")
plt.colorbar(label="Normalized score")
plt.title("Hardware-Counter Summary Matrix")
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_06_hardware_counter_overlays.md"

summary_cols = [
    "distribution", "regime", "observed_throughput_mib_s", "observed_latency_ns",
    "branch_miss_rate", "cache_miss_rate", "ipc", "cycles_per_item",
    "fragmentation_score", "counter_pressure_score", "abs_fragmentation_counter_gap"
]
summary_cols = [c for c in summary_cols if c in merged.columns]

lines = [
    "# Report 06 — Hardware-Counter Overlays",
    "",
    "This report overlays hardware-counter metrics onto RML phase and benchmark results.",
    "",
    "Constraint view:",
    "> performance becomes interpretable when structural predictions meet hardware counters.",
    "",
    "## Generated outputs",
    "",
    f"- Metrics CSV: `{csv_path}`",
    f"- Metrics JSON: `{json_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    "",
    "## Hardware-counter overlay summary",
    "",
    merged[summary_cols].to_markdown(index=False),
    "",
    "## Interpretation",
    "",
    "- Branch miss rate and cache miss rate help convert structural proxies into measurable systems behavior.",
    "- Counter pressure can validate, correct, or refine the fragmentation model from Notebook 04.",
    "- IPC vs throughput helps distinguish hardware saturation from structural coherence.",
    "- Real counters are the next bridge from RML phase maps to architecture-aware performance claims.",
    "",
    "## Next step",
    "",
    "Notebook 07 can build adaptive path-selection rules using structural metrics plus observed counters.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook06_hardware_counter_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook06_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_06_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))